**Ex 1**

In [7]:
import numpy as np

# Each row: [semi-major axis (a), flattening (f), ...]
ellipsoid_params = np.array([
    [6376985, 1/308.6, 0, 0, 0],
    [6377276, 1/300.8, 0, 0, 0],
    [6377397, 1/299.1528128, 0, 0, 0],
    [6378338, 1/288.5, 0, 0, 0],
    [6378249, 1/293.5, 0, 0, 0],
    [6378140, 1/298.3, 0, 0, 0],
    [6378388, 1/297.0, 0, 0, 0],
    [6378245, 1/298.3, 0, 0, 0],
    [6378137, 1/298.257223563, 0, 0, 0]
])

ellipsoid_params[:, 2] = ellipsoid_params[:, 0] * (1 - ellipsoid_params[:, 1])  # Calculate semi-minor axis (c) = a * (1 - f)
# eccentricity squared (e2)
ellipsoid_params[:, 3] = ellipsoid_params[:, 1] * (2 - ellipsoid_params[:, 1])  # e2 = f * (2 - f)
# second eccentricity squared (e2_prime)
ellipsoid_params[:, 4] = ellipsoid_params[:, 3] / (1 - ellipsoid_params[:, 3])  # e2_prime = e2 / (1 - e2)

print("Ellipsoid Parameters:")
for i, params in enumerate(ellipsoid_params):
    print(f"Ellipsoid {i+1}: a = {params[0]:.2f}, f = {params[1]:.6f}, c = {params[2]:.2f}, e2 = {params[3]:.6f}, e2_prime = {params[4]:.6f}")

Ellipsoid Parameters:
Ellipsoid 1: a = 6376985.00, f = 0.003240, c = 6356320.76, e2 = 0.006470, e2_prime = 0.006513
Ellipsoid 2: a = 6377276.00, f = 0.003324, c = 6356074.95, e2 = 0.006638, e2_prime = 0.006682
Ellipsoid 3: a = 6377397.00, f = 0.003343, c = 6356078.81, e2 = 0.006674, e2_prime = 0.006719
Ellipsoid 4: a = 6378338.00, f = 0.003466, c = 6356229.38, e2 = 0.006920, e2_prime = 0.006969
Ellipsoid 5: a = 6378249.00, f = 0.003407, c = 6356517.32, e2 = 0.006803, e2_prime = 0.006849
Ellipsoid 6: a = 6378140.00, f = 0.003352, c = 6356758.37, e2 = 0.006693, e2_prime = 0.006739
Ellipsoid 7: a = 6378388.00, f = 0.003367, c = 6356911.95, e2 = 0.006723, e2_prime = 0.006768
Ellipsoid 8: a = 6378245.00, f = 0.003352, c = 6356863.02, e2 = 0.006693, e2_prime = 0.006739
Ellipsoid 9: a = 6378137.00, f = 0.003353, c = 6356752.31, e2 = 0.006694, e2_prime = 0.006739


**Ex 2**

In [18]:
# Convert DMS (degrees, minutes, seconds) to decimal degrees
def dms_to_decimal(degrees, minutes, seconds):
    return degrees + minutes/60 + seconds/3600

geodetic_coordinates = np.array([
    [dms_to_decimal(44, 45, 1.03930), dms_to_decimal(7, 24, 29.20335), 322.490],
    [dms_to_decimal(44, 45, 1.03930), dms_to_decimal(7, 24, 29.20335), 2322.490],
    [dms_to_decimal(44, 47, 10.90505), dms_to_decimal(7, 30, 26.53939), 305.736]
])

# convert to X,Y,Z

# Convert degrees to radians for latitude and longitude
lat = np.deg2rad(geodetic_coordinates[:, 0])
lon = np.deg2rad(geodetic_coordinates[:, 1])
h = geodetic_coordinates[:, 2]

a_WGS84 = ellipsoid_params[-1,0]
e2_WGS84 = ellipsoid_params[-1, 3]
a_Hayford = ellipsoid_params[6, 0]
e2_Hayford = ellipsoid_params[6, 3]

W_WGS84 =  np.sqrt(1 - e2_WGS84 * np.sin(lat)**2)
W_Hayford = np.sqrt(1 - e2_Hayford * np.sin(lat)**2)

# Use already defined W_WGS84, W_Hayford, a_WGS84, a_Hayford, e2_WGS84, e2_Hayford

x_WGS84 = (a_WGS84 / W_WGS84 + h) * np.cos(lat) * np.cos(lon)
y_WGS84 = (a_WGS84 / W_WGS84 + h) * np.cos(lat) * np.sin(lon)
z_WGS84 = (a_WGS84 / W_WGS84 * (1 - e2_WGS84) + h) * np.sin(lat)

x_Hayford = (a_Hayford / W_Hayford + h) * np.cos(lat) * np.cos(lon)
y_Hayford = (a_Hayford / W_Hayford + h) * np.cos(lat) * np.sin(lon)
z_Hayford = (a_Hayford / W_Hayford * (1 - e2_Hayford) + h) * np.sin(lat)

print("WGS84 X:", x_WGS84)
print("WGS84 Y:", y_WGS84)
print("WGS84 Z:", z_WGS84)
print("Hayford X:", x_Hayford)
print("Hayford Y:", y_Hayford)
print("Hayford Z:", z_Hayford)

WGS84 X: [4499525.42646913 4500933.9342329  4495694.26904015]
WGS84 Y: [585034.12922753 585217.26523299 592457.86038784]
WGS84 Z: [4467910.35890505 4469318.39551073 4470744.77760491]
Hayford X: [4499734.13872965 4501142.64649342 4495902.84444191]
Hayford Y: [585061.26626622 585244.40227168 592485.34716271]
Hayford Z: [4467990.35599916 4469398.39260483 4470824.86573938]


In [22]:
# differences between WGS84 and Hayford
diff_x = x_WGS84 - x_Hayford
diff_y = y_WGS84 - y_Hayford
diff_z = z_WGS84 - z_Hayford
print("Differences (WGS84 - Hayford):")
print("Diff X:", diff_x)
print("Diff Y:", diff_y)
print("Diff Z:", diff_z)

# cartesian coordinate differences between the first and second points
diff_coords = x_WGS84[0] - x_WGS84[1], y_WGS84[0] - y_WGS84[1], z_WGS84[0] - z_WGS84[1]
print("Cartesian coordinate differences between the first and second points:")
print("Diff X:", diff_coords[0])
print("Diff Y:", diff_coords[1])
print("Diff Z:", diff_coords[2])

Differences (WGS84 - Hayford):
Diff X: [-208.71226052 -208.71226052 -208.57540176]
Diff Y: [-27.13703869 -27.13703869 -27.48677487]
Diff Z: [-79.9970941  -79.9970941  -80.08813447]
Cartesian coordinate differences between the first and second points:
Diff X: -1408.5077637666836
Diff Y: -183.13600546354428
Diff Z: -1408.0366056719795


**Ex 3**

In [44]:
from matplotlib import pyplot as plt

cartesian_coordinates = [
    [4498329.37, 562840.77, 4472537.61],
    [4495694.27, 592457.86, 4470744.78],
    [4503484.72, 578160.75, 4465024.30],
    [4499525.43, 585034.13, 4467910.36]
]

def iterative_conversion(x, y, z, a, e2):
    lon = np.arctan2(y, x)
    r = np.sqrt(x**2 + y**2)
    hs = []
    lats = []
    h = 0.0
    lat = np.arctan2(z, r)
    lats.append(lat)

    while True:
        old_lat = lat
        old_h = h

        W = np.sqrt(1 - e2 * np.sin(lat)**2)
        N = a / W

        h = x / (np.cos(lat) * np.cos(lon)) - N
        hs.append(h)
        lat = np.arctan2(z, (r * (1 - e2 * N / (N + h))))
        lats.append(lat)

        if lat - old_lat < 1e-8 and h - old_h < -1e-8:
            break

    # plot the trend of lat
    # plt.plot(np.rad2deg(lats), label='Latitude (lat)')
    # plt.xlabel('Iteration')
    # plt.ylabel('Value')
    # plt.title('Convergence of Latitude')
    # plt.legend()
    # plt.grid()
    # plt.show()

    # plot the trend of h
    # plt.plot(hs, label='Height (h)')
    # plt.xlabel('Iteration')
    # plt.ylabel('Value')
    # plt.title('Convergence of Height')
    # plt.grid()
    # plt.show()
    
    # print heights of hs
    print("Heights (h) during iterations:", hs)


    return np.rad2deg(lat), np.rad2deg(lon), h

for coords in cartesian_coordinates:
    x, y, z = coords
    lat, lon, h = iterative_conversion(x, y, z, a_WGS84, e2_WGS84)
    print(f"Converted Cartesian coordinates {coords} to Geodetic: Lat = {lat:.6f}, Lon = {lon:.6f}, h = {h:.2f}")
    lat, lon, h = iterative_conversion(x, y, z, a_Hayford, e2_Hayford)
    print(f"Converted Cartesian coordinates {coords} to Geodetic (Hayford): Lat = {lat:.6f}, Lon = {lon:.6f}, h = {h:.2f}")

Heights (h) during iterations: [np.float64(-20385.072028251365), np.float64(816.6710453247651), np.float64(745.7240187870339), np.float64(745.9606331493706), np.float64(745.9598440118134)]
Converted Cartesian coordinates [4498329.37, 562840.77, 4472537.61] to Geodetic: Lat = 44.805162, Lon = 7.131909, h = 745.96
Heights (h) during iterations: [np.float64(-20681.21070423629), np.float64(611.5776244550943), np.float64(540.0183973386884), np.float64(540.2580800577998), np.float64(540.257277247496)]
Converted Cartesian coordinates [4498329.37, 562840.77, 4472537.61] to Geodetic (Hayford): Lat = 44.805984, Lon = 7.131909, h = 540.26
Heights (h) during iterations: [np.float64(-20811.231448834762), np.float64(376.36038285307586), np.float64(305.5029359553009), np.float64(305.739111286588), np.float64(305.7383240805939)]
Converted Cartesian coordinates [4495694.27, 592457.86, 4470744.78] to Geodetic: Lat = 44.786363, Lon = 7.507372, h = 305.74
Heights (h) during iterations: [np.float64(-21107.

**Ex 4**

In [54]:
import pandas as pd

file_path = "points_helmert.txt" 
data = pd.read_csv(file_path, comment='%', delim_whitespace=True, header=None)

XA = data[[0, 1, 2]].to_numpy()  # Datum A (ETRF89)
XB = data[[3, 4, 5]].to_numpy()  # Datum B (IGS05)

XA_norm = XA / 1e6
l0 = (XB - XA).reshape(-1, 1) 

n = XA.shape[0]
A = np.zeros((n * 3, 7))

for i in range(n):
    X, Y, Z = XA_norm[i]
    A[3*i:3*i+3, :] = [
        [1, 0, 0, 0,     0,  Z, -Y],
        [0, 1, 0, -Z,    0,  0,  X],
        [0, 0, 1,  Y, -X,  0,  0]
    ]

x_hat = np.linalg.inv(A.T @ A) @ A.T @ l0
residuals = A @ x_hat - l0

param_labels = ['ΔTx [m]', 'ΔTy [m]', 'ΔTz [m]', 'Scale [ppm]', 'Rx', 'Ry', 'Rz']
print("=== Estimated Helmert Parameters ===")
for label, val in zip(param_labels, x_hat.flatten()):
    print(f"{label:<10}: {val:.6f}")

print("\n=== Residuals (in meters) ===")
for i in range(n):
    r = residuals[3*i:3*i+3].flatten()
    print(f"Point {i+1:2d}: dX={r[0]:+.6f}  dY={r[1]:+.6f}  dZ={r[2]:+.6f}")

rmse = np.sqrt(np.mean(residuals**2))
print(f"\nRMSE of residuals: {rmse:.6f} m")

=== Estimated Helmert Parameters ===
ΔTx [m]   : -0.876298
ΔTy [m]   : 0.943807
ΔTz [m]   : -2.381511
Scale [ppm]: 0.391948
Rx        : -0.535524
Ry        : 0.164526
Rz        : 0.260450

=== Residuals (in meters) ===
Point  1: dX=-0.000804  dY=+0.011036  dZ=+0.001153
Point  2: dX=-0.000471  dY=+0.013263  dZ=-0.000014
Point  3: dX=+0.010233  dY=-0.004437  dZ=-0.001472
Point  4: dX=+0.007637  dY=-0.009710  dZ=-0.002583
Point  5: dX=+0.012393  dY=-0.005517  dZ=-0.001075
Point  6: dX=+0.004607  dY=-0.011013  dZ=+0.005772
Point  7: dX=-0.008032  dY=-0.000165  dZ=-0.000201
Point  8: dX=-0.022783  dY=+0.007005  dZ=+0.000021
Point  9: dX=+0.002659  dY=+0.018815  dZ=+0.012042
Point 10: dX=-0.003853  dY=-0.015219  dZ=-0.001233
Point 11: dX=-0.007352  dY=-0.008263  dZ=-0.002044
Point 12: dX=+0.005768  dY=+0.004204  dZ=-0.010366

RMSE of residuals: 0.008529 m


/tmp/ipykernel_84222/2881013093.py:4: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  data = pd.read_csv(file_path, comment='%', delim_whitespace=True, header=None)
